In [70]:
from bs4 import BeautifulSoup
import json
import pandas as pd
import re
import requests


In [15]:
root = "https://immovlan.be/en"
end_point = "real-estate"
params_first_page = "transactiontypes=for-sale&propertytypes=house,apartment&islifeannuity=no&noindex=1"
try:
    response = requests.get(f"{root}/{end_point}?{params_first_page}", headers={"User-Agent": "max-exercice"})
    
except BaseException as ex:
    print(0, ex)
else:
    print(1, response.status_code)



1 200


In [21]:
soup = BeautifulSoup(response.content, "html")

titles = soup.find_all("h2", attrs={"class": "card-title"})
all_titles = soup.find_all("h2")
print(titles == all_titles)


True


In [24]:
link_titles = soup.select('h2 a')
print(f"Number of links per page: {len(link_titles)}")
for l in link_titles:
    print(l.get("href"))

Number of links per page: 20
https://immovlan.be/en/projectdetail/1132102-1049619
https://immovlan.be/en/projectdetail/2023631-7710858
https://immovlan.be/en/detail/residence/for-sale/8930/rekkem/rbw13670
https://immovlan.be/en/detail/residence/for-sale/1820/steenokkerzeel/vbe32011
https://immovlan.be/en/detail/residence/for-sale/6280/gerpinnes/vwd17735
https://immovlan.be/en/detail/apartment/for-sale/6880/bertrix/vbe31926
https://immovlan.be/en/detail/apartment/for-sale/8400/oostende/rbw13583
https://immovlan.be/en/detail/residence/for-sale/2300/turnhout/rbw13423
https://immovlan.be/en/detail/mixed-building/for-sale/1000/brussels/vbe31836
https://immovlan.be/en/detail/mixed-building/for-sale/5500/dinant/vbe31835
https://immovlan.be/en/detail/residence/for-sale/2840/rumst/rbw13365
https://immovlan.be/en/detail/mixed-building/for-sale/1060/sint-gillis/vbe31787
https://immovlan.be/en/detail/bungalow/for-sale/8800/roeselare/rbw13280
https://immovlan.be/en/detail/residence/for-sale/2980/zo

In [25]:

# first looking at the first 10 pages (~200 properties)
links_p2_p10 = []
with requests.Session() as session:
    for i in range(2, 11):
        params_next_page = f"transactiontypes=for-sale&propertytypes=house,apartment&islifeannuity=no&page={i}&noindex=1"

        response = session.get(f"{root}/{end_point}?{params_next_page}", headers={"User-Agent": "max-exercice"})
        link_titles = soup.select('h2 a')
        print(f"Number of links in page {i}: {len(link_titles)}")
        for l in link_titles:
            links_p2_p10.append(l.get("href"))
        



Number of links in page 2: 20
Number of links in page 3: 20
Number of links in page 4: 20
Number of links in page 5: 20
Number of links in page 6: 20
Number of links in page 7: 20
Number of links in page 8: 20
Number of links in page 9: 20
Number of links in page 10: 20


In [28]:
print(len(links_p2_p10), "links : ")
print(*links_p2_p10, sep="\n")


180 links : 
https://immovlan.be/en/projectdetail/1132102-1049619
https://immovlan.be/en/projectdetail/2023631-7710858
https://immovlan.be/en/detail/residence/for-sale/8930/rekkem/rbw13670
https://immovlan.be/en/detail/residence/for-sale/1820/steenokkerzeel/vbe32011
https://immovlan.be/en/detail/residence/for-sale/6280/gerpinnes/vwd17735
https://immovlan.be/en/detail/apartment/for-sale/6880/bertrix/vbe31926
https://immovlan.be/en/detail/apartment/for-sale/8400/oostende/rbw13583
https://immovlan.be/en/detail/residence/for-sale/2300/turnhout/rbw13423
https://immovlan.be/en/detail/mixed-building/for-sale/1000/brussels/vbe31836
https://immovlan.be/en/detail/mixed-building/for-sale/5500/dinant/vbe31835
https://immovlan.be/en/detail/residence/for-sale/2840/rumst/rbw13365
https://immovlan.be/en/detail/mixed-building/for-sale/1060/sint-gillis/vbe31787
https://immovlan.be/en/detail/bungalow/for-sale/8800/roeselare/rbw13280
https://immovlan.be/en/detail/residence/for-sale/2980/zoersel/rbw12783
h

In [45]:
first_link = link_titles[0].get("href")
response = requests.get(first_link, headers={"User-Agent": "max-exercice"})
print(response.status_code)

200


In [54]:
def finding_purple_properties(soup):
    properties = soup.find_all("li", attrs={"class": "property-highlight"})
    proper_res = []
    if not properties:
        properties = soup.find_all("div", attrs={"class": "property-highlight"})

    bed_already_seen = False
    for p in properties:
        stripped = " ".join([elem.strip().replace("\u202f", " ") for elem in p.text.split("\n") if not all([char == " " for char in elem])])

        if "Bed" in stripped:
            print("found bed")
            if bed_already_seen:
                break
            else:
                bed_already_seen = True
        proper_res.append(stripped)
    return proper_res

In [56]:
soup = BeautifulSoup(response.content, "html")
city = soup.find("span", attrs={"class": "city-line"})
price = soup.find("span", attrs={"class": "detail__header_price_data"})
print(city.text)
print(price.text.strip())
properties = soup.find_all("li", attrs={"class": "property-highlight"})
proper_res = finding_purple_properties(soup)
print(proper_res)



3020 Winksele
475 000 € - 505 000 €
found bed
found bed
['from 475 000 € to 505 000 €', '3 Bedroom(s)', '23 unit(s)']


In [57]:
other_link = links_p2_p10[-2]
response = requests.get(other_link, headers={"User-Agent": "max-exercice"})
print(response.status_code)
soup1 = BeautifulSoup(response.content, "html")
city = soup1.find("span", attrs={"class": "city-line"})
price = soup1.find("span", attrs={"class": "detail__header_price_data"})
print(city.text)
print(price.text.strip())

proper_res = finding_purple_properties(soup1)
print(proper_res)


200
5060 Auvelais
From 179 000 €
found bed
['3 Bedroom(s)', '98 m²', '1 Garage', '1 Bathroom(s)']


In [69]:
def dic_from_more_info(more_info):
    dic_infos = {}
    h4s = [h.text.replace("\n", "").strip() for h in more_info.find_all("h4")]
    ps = [p.text.replace("\n", "").strip() for p in more_info.find_all("p")]
    for i, title in enumerate(h4s):
        dic_infos[title] = ps[i]
    return dic_infos

more_info = soup1.find("div", attrs={"class": "general-info-wrapper"})
if more_info:
    dic_infos = dic_from_more_info(more_info)
dic_infos

{'State of the property': 'Excellent',
 'Currently leased': 'No',
 'Number of bedrooms': '3',
 'Surface bedroom 1': '17 m²',
 'Surface bedroom 2': '13 m²',
 'Surface bedroom 3': '12 m²',
 'Livable surface': '98 m²',
 'Furnished': 'No',
 'Surface of living-room': '16 m²',
 'Cellar': 'Yes',
 'Garage': 'Yes',
 'Number of garages': '1',
 'Kitchen equipment': 'Super equipped',
 'Surface kitchen': '17 m²',
 'Number of bathrooms': '1',
 'Number of toilets': '2',
 'Type of heating': 'Gas',
 'Type of glazing': 'Double glass',
 'Elevator': 'No',
 'Access for disabled': 'No',
 'Number of facades': '2',
 'Number of floors': '3',
 'Garden': 'Yes',
 'Surface garden': '200 m²',
 'Terrace': 'Yes',
 'Surface terrace': '20 m²',
 'Total land surface': '236 m²',
 'Sewer Connection': 'Yes',
 'Gas': 'Yes',
 'Running water': 'Yes',
 'Specific primary energy consumption': '344 kWh/m²/year',
 'Yearly total primary energy consumption': '39300 kWh/year',
 'Validity date EPC/PEB': '06/09/2034',
 'CO2 emission': '

In [75]:
df = pd.DataFrame([dic_infos])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 40 columns):
 #   Column                                   Non-Null Count  Dtype
---  ------                                   --------------  -----
 0   State of the property                    1 non-null      str  
 1   Currently leased                         1 non-null      str  
 2   Number of bedrooms                       1 non-null      str  
 3   Surface bedroom 1                        1 non-null      str  
 4   Surface bedroom 2                        1 non-null      str  
 5   Surface bedroom 3                        1 non-null      str  
 6   Livable surface                          1 non-null      str  
 7   Furnished                                1 non-null      str  
 8   Surface of living-room                   1 non-null      str  
 9   Cellar                                   1 non-null      str  
 10  Garage                                   1 non-null      str  
 11  Number of garages    